In [1]:
# Verificar entorno
import sys
print(sys.executable)

/opt/anaconda3/bin/python


In [51]:
#!pip install geopandas

#### 1. Librerías y conexión GEE

In [2]:
# Librerías
import ee
import pandas as pd
from pathlib import Path

#### Mecanismo de autorización
1. Al ejecutar las siguientes líneas se habilitará un link de acceso
2. Asegura los permisos apropiados para google Earth Engine y demás necesarios
3. Se genera un Token
4. Copiar y pegar el token en la ventana emergente que aparece en la ventana superior de VSC

* Para habilitar nuevos usuarios es necesario desmarcar "ee.Authenticate(force=True)" - Actualmente ya tiene habilitado el ingreso del usuario presente.  
(4/1AeoWuM9_9T1gCVAvTELP8S2YXcFW26H8_p7npV3xK4wc_OUw6Fb5i68ThS8)

In [4]:
import ee
ee.Authenticate(force=True)

Enter verification code:  4/1AeoWuM9vKe0BQXMIW5FwPGvyRUr_0Po0I9ybj1ND0V6EOkvQs100L6KmimE



Successfully saved authorization token.


In [5]:
# Inicializar GEE
ee.Reset()
ee.Initialize(project="indice-climatico-cafe")

print(ee.String("Conexión exitosa con proyecto: indice-climatico-cafe").getInfo())

Conexión exitosa con proyecto: indice-climatico-cafe


#### 2.Cargar municipios oficiales desde asset

In [6]:
# Cargar municipios oficiales de Caldas
municipios_ee = ee.FeatureCollection(
    "projects/indice-climatico-cafe/assets/municipios_caldas"
)

# Validar
print("Número de municipios:", municipios_ee.size().getInfo())
print("Columnas:", municipios_ee.first().propertyNames().getInfo())

Número de municipios: 27
Columnas: ['municipio', 'system:index']


#### 3. Fechas de análisis

In [7]:
# Periodo de análisis
fecha_inicio = "2005-01-01"
fecha_fin = "2025-12-31"

print("Periodo:", fecha_inicio, "a", fecha_fin)

Periodo: 2005-01-01 a 2025-12-31


#### 4. Variables ERA5-Land ampliadas

In [8]:
variables_era5 = [
    "temperature_2m",
    "temperature_2m_min",
    "temperature_2m_max",
    "dewpoint_temperature_2m",
    "total_precipitation_sum",
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
    "surface_solar_radiation_downwards_sum",
    "potential_evaporation_sum",
    "total_evaporation_sum",
    "surface_runoff_sum",
    "sub_surface_runoff_sum",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "surface_pressure"
]

era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .filterDate(fecha_inicio, fecha_fin)
    .select(variables_era5)
)

print("Número de imágenes ERA5:", era5.size().getInfo())

Número de imágenes ERA5: 7669


| Variable                              | Descripción                                                                 |
|--------------------------------------|-----------------------------------------------------------------------------|
| temperature_2m_min                   | Temperatura mínima diaria del aire a 2 metros de altura.                   |
| temperature_2m_max                   | Temperatura máxima diaria del aire a 2 metros de altura.                   |
| dewpoint_temperature_2m              | Temperatura a la que el aire se satura de humedad (indicador de humedad). |
| total_precipitation_sum              | Cantidad total de lluvia acumulada en el día.                              |
| volumetric_soil_water_layer_1        | Cantidad de agua en la capa superficial del suelo (0–7 cm).               |
| volumetric_soil_water_layer_2        | Cantidad de agua en una capa más profunda del suelo (7–28 cm).            |
| surface_solar_radiation_downwards_sum| Energía solar que llega a la superficie terrestre.                        |
| potential_evaporation_sum            | Cantidad de agua que podría evaporarse dadas las condiciones climáticas.  |
| total_evaporation_sum                | Cantidad real de agua que se evapora desde el suelo y la vegetación.      |
| surface_runoff_sum                   | Agua que se escurre por la superficie del suelo después de la lluvia.     |
| sub_surface_runoff_sum               | Agua que se infiltra y se mueve por debajo del suelo.                     |
| u_component_of_wind_10m              | Componente del viento en dirección este-oeste a 10 metros.                |
| v_component_of_wind_10m              | Componente del viento en dirección norte-sur a 10 metros.                |
| surface_pressure                     | Presión atmosférica en la superficie terrestre.                           |

In [9]:
import geemap

Map = geemap.Map(center=[5.25, -75.45], zoom=8)

Map.addLayer(
    municipios_ee.style(**{
        "color": "blue",
        "fillColor": "00000000",
        "width": 1
    }),
    {},
    "Municipios de Caldas"
)

Map

Map(center=[5.25, -75.45], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

#### 5. Agregación mensual GEE

In [10]:
# Crear lista de meses entre fecha_inicio y fecha_fin
inicio = ee.Date(fecha_inicio)
fin = ee.Date(fecha_fin)

n_meses = fin.difference(inicio, "month").toInt()

fechas_mensuales = ee.List.sequence(0, n_meses.subtract(1)).map(
    lambda n: inicio.advance(n, "month")
)

print("Número de meses:", fechas_mensuales.size().getInfo())

Número de meses: 251


In [11]:
def reducir_mensual(fecha):
    fecha = ee.Date(fecha)
    fin = fecha.advance(1, "month")
    
    # Filtrar imágenes del mes
    coleccion_mes = era5.filterDate(fecha, fin)
    
    # Promedio mensual
    img = coleccion_mes.mean().set("system:time_start", fecha.millis())
    
    # Reducir por municipio
    reduccion = img.reduceRegions(
        collection=municipios_ee,
        reducer=ee.Reducer.mean(),
        scale=11132
    )
    
    # Limpiar salida
    def limpiar(feat):
        props = {
            "municipio": feat.get("municipio"),
            "date": fecha.format("YYYY-MM-dd")
        }
        
        for var in variables_era5:
            props[var] = feat.get(var)
        
        return ee.Feature(None, props)
    
    return reduccion.map(limpiar)

In [12]:
clima_mensual = ee.FeatureCollection(
    fechas_mensuales.map(reducir_mensual)
).flatten()

print("Colección mensual creada")

Colección mensual creada


In [13]:
task = ee.batch.Export.table.toDrive(
    collection=clima_mensual,
    description="clima_mensual_caldas_2005_2025",
    folder="GEE_exports",
    fileNamePrefix="clima_mensual_caldas_2005_2025",
    fileFormat="CSV",
    selectors=[
        "municipio",
        "date"
    ] + variables_era5
)

task.start()

print("Exportación mensual iniciada ")
print("Revisar la pestaña Tasks en GEE")

Exportación mensual iniciada 
Revisar la pestaña Tasks en GEE


### Descarga variable NDVI

In [14]:
# Colección MODIS NDVI
ndvi_modis = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
    .filterDate(fecha_inicio, fecha_fin)
    .select("NDVI")
)

# Escalar NDVI
def escalar_ndvi(image):
    ndvi = image.multiply(0.0001).rename("ndvi")
    return ndvi.copyProperties(image, ["system:time_start", "system:index"])

ndvi_modis = ndvi_modis.map(escalar_ndvi)

print("Número de imágenes ndvi_modis:", ndvi_modis.size().getInfo())

Número de imágenes ndvi_modis: 483


In [15]:
def reducir_ndvi_mensual(fecha):
    fecha = ee.Date(fecha)
    fin = fecha.advance(1, "month")
    
    # Usar ndvi_modis (ya escalado)
    coleccion_mes = ndvi_modis.filterDate(fecha, fin)
    
    img = coleccion_mes.mean().set("system:time_start", fecha.millis())
    
    reduccion = img.reduceRegions(
        collection=municipios_ee,
        reducer=ee.Reducer.mean(),
        scale=250
    )
    
    def limpiar(feat):
        return ee.Feature(None, {
            "municipio": feat.get("municipio"),
            "date": fecha.format("YYYY-MM-dd"),
            "ndvi_mean": feat.get("mean")
        })
    
    return reduccion.map(limpiar)

In [16]:
ndvi_mensual = ee.FeatureCollection(
    fechas_mensuales.map(reducir_ndvi_mensual)
).flatten()

print("NDVI mensual listo")

NDVI mensual listo


In [17]:
task = ee.batch.Export.table.toDrive(
    collection=ndvi_mensual,
    description="ndvi_mensual_caldas_2005_2025",
    folder="GEE_exports",
    fileNamePrefix="ndvi_mensual_caldas_2005_2025",
    fileFormat="CSV"
)

task.start()

print("Export NDVI iniciado ")

Export NDVI iniciado 
